In [ ]:
# %%
# polars script with datatype toggles
# read_csv
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import os
import re
import logging
from tqdm import tqdm
 

# %%
# variables
output_fname = "processed_tab_eskd_v4.csv"
icd_file = "/opt/data/commonfilesharePHI/ldiao/ckd_project/icd_mapping.csv"
subset = False # <<

if subset: 
    subset_size = "10000"  # 10, 100, full # <<
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}"
    # output_dir = f"/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_subset_{subset_size}"
    event_file =  f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}/unprocessed_tab_subset_{subset_size}.csv"
if not subset:
    # output_dir = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_full"
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_full"
    # event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"
    event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v3.rpt.parquet"

# filter ckd stage
filter_ckd_stage = True # <<

try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
except FileExistsError:
    print(f"Output directory already exists: {output_dir}")

print(f"Processing started. Output directory: {output_dir}")

# %%
# Setup logging
log_file_path = os.path.join(output_dir, "tab_gen_m.log")
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"Processing started. Output directory: {output_dir}")

# %%
# -----------------------------
# Load and preprocess with Polars
# -----------------------------
# Using pl.read_csv to load the entire file into a DataFrame
# Add a toggle to switch between separators
print(event_file)
if subset: 
    df = pl.read_csv(
        event_file,
        separator='$',
        infer_schema_length=None,
        null_values="null",
    ).unique()
    # df = df.with_columns(
#     pl.col("PatientID").cast(pl.Utf8, strict=False),
#     pl.col("EventTimeStamp").cast(pl.Utf8, strict=False),
#     pl.col("DataCategory").cast(pl.Utf8, strict=False),
#     pl.col("DataType").cast(pl.Utf8, strict=False),
# )

if not subset: 
    csv = event_file 
    df = pl.read_parquet(csv)
    print(len(df['PatientID'].unique()))

logger.info(f"Initial DataFrame schema: {df.schema}")

In [ ]:
print(df.shape)

In [ ]:
# -----------------------------
# ICD code -> long title mapping (ported from embedding_gen_pl_v3.py)
# -----------------------------
icd_map_df = pl.read_csv(icd_file)
icd_map_df = icd_map_df.with_columns(
    pl.col("icd_code").cast(pl.Utf8).str.replace(".", "", literal=True)
)
icd_map = dict(zip(icd_map_df["icd_code"], icd_map_df["long_title"]))
logger.info(f"Loaded {len(icd_map)} ICD code -> long_title mappings from {icd_file}")

# Clean and prepare initial columns
df = df.with_columns(
    pl.col("EventTimeStamp").str.strptime(pl.Datetime("us")).dt.date().alias("EventDate")
)


In [ ]:
icd_map_df.shape

In [ ]:
len(icd_map.keys())

In [ ]:
df.shape

In [ ]:
# Clean and prepare initial columns
df = df.with_columns(
    pl.col("EventTimeStamp").str.to_datetime("%Y-%m-%d %H:%M:%S%.f", strict=False).alias("EventTimeStamp"),
    pl.col("DataCategory").fill_null("None"),
).with_columns(
    pl.col("EventTimeStamp").dt.date().alias("EventDate")
)



In [ ]:
df

In [ ]:
# -----------------------------
# Base: full patient-day index
# -----------------------------
all_days = df.select(["PatientID", "EventDate"]).unique().sort(["PatientID", "EventDate"])
# all_days = all_days.drop_nulls("EventDate")

print(len(all_days['PatientID'].unique()))
all_days.shape

In [ ]:
# Extract and forward-fill ICD

custom_map = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,  # ESRD
    9: 0,  # CKD, unspecified stage
}


In [ ]:
print("Building Filter")
ckd_icd_df = (
    df.filter(pl.col("DataCategory").str.contains("N18"))
    .with_columns(
      pl.col("DataCategory")
        .str.extract(r"N18\.([1-9])", 1)
        .cast(pl.Int64)
        .replace(custom_map, default=None)
        .alias("CKD_stage_numeric")
    )
    .select([pl.col("PatientID"), pl.col("EventTimeStamp"), pl.col("EventDate"), pl.col("DataCategory"),pl.col("CKD_stage_numeric")])
    .with_columns(
        pl.col("CKD_stage_numeric")
            .max()
            .over("PatientID")
            .alias("max_stage")
    )
    .filter(pl.col("max_stage") >= 3)
    .unique()
) 

In [ ]:
ckd_icd_df

In [ ]:
ckd_icd_df.shape

In [ ]:
ckd_icd_df 

In [ ]:
ckd_icd_df

In [ ]:
icd_daywise = (
    ckd_icd_df.sort("PatientID", "EventDate")
    .group_by("PatientID", "EventDate")
    .agg([
        pl.col("DataCategory").first().alias("ICD_combined"),
        pl.col("CKD_stage_numeric").max().alias("CKD_stage_numeric"),
    ])
)
base_df = all_days.join(icd_daywise, on=["PatientID", "EventDate"], how="left").sort(["PatientID", "EventDate"])


In [ ]:
print("Filtering and converting")
df = (
    df
    .join(
        ckd_icd_df.select("PatientID").unique(), 
        on="PatientID", 
        how='inner')
    .drop_nulls(subset=["DataNumeric"])
    .with_columns([
        
        pl.col("DataCategory").fill_null(pl.col("META_2"))
    ])
    

)
print(len(df['PatientID'].unique()))
# %% demographics and encounter


In [ ]:
df.shape

In [ ]:
# %%
aki_icd_codes = ["N17.0", "N17.1", "N17.2", "N17.8", "N17.9"]
aki_events = df.filter(
    (pl.col("DataType") == "Diagnosis") & 
    (pl.col("DataCategory").is_in(aki_icd_codes))
)

# 3. Sum (count) the occurrences into one column per Patient/Date
aki_count = aki_events.group_by(["PatientID", "EventDate"]).agg([
    pl.len().alias("AKI_ICD_Total"),
])

In [ ]:
print(base_df.schema)
print(aki_count.schema)  # and lab_pivot.schema


In [ ]:


# 4. Join this single column back to your base_df
base_df = base_df.join(aki_count, on=["PatientID", "EventDate"], how="left", join_nulls=True).with_columns(
    pl.col("AKI_ICD_Total").fill_null(0) # Ensure days with no AKI are 0, not null
)

# %%
base_df.head()


In [ ]:
# %%
# top lab features from the paper ---
top_lab_features = [
    "CREATININE", "GFR", "GFREST", "ALBUMIN/CREATININE RATIO", 
    "PROTEIN/CREATININE RATIO", "BUN", "PTH"
]


lab_df = df.filter(
    (pl.col("DataType") == "Labs") & 
    # (pl.col("DataNumeric").is_not_null()) &
    (pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().str.contains("|".join(top_lab_features)))
).with_columns(
    pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().alias("LabCategory")
).group_by("PatientID", "EventDate", "LabCategory").agg(pl.col("DataNumeric").first())

# %%
lab_df

# %%
lab_pivot = lab_df.pivot(
    index=["PatientID", "EventDate"],
    on="LabCategory",
    values="DataNumeric",
    aggregate_function="first",
)

# %%
lab_pivot 


In [ ]:
print(base_df.shape)

In [ ]:
# %%
# Dynamically generate a dictionary for renaming the pivoted columns
rename_dict = {c: f"lab_{c}" for c in lab_pivot.columns[2:]}
lab_pivot = lab_pivot.rename(rename_dict)

base_df = base_df.join(lab_pivot, on=["PatientID", "EventDate"], how="left", join_nulls=True)


In [ ]:
# base_df =base_df.join(
#     ckd_icd_df.select(["PatientID", "META_1", "CKD_stage_numeric", "max_stage"])
#     , on=["PatientID", "META_1"], how='left'
# )

In [ ]:
print(base_df.shape)

In [ ]:
# -----------------------------
# Final report
# -----------------------------
logger.info(f"[INFO] Final tabular shape: {base_df.shape}")
logger.info(f"[INFO] Sample features:\n{base_df.head()}")
logger.info(f"[INFO] CKD stage counts:\n{base_df['CKD_stage_numeric'].value_counts(sort=True)}")
base_df_path = os.path.join(output_dir, output_fname)
logger.info(f"Writing final DataFrame of shape {base_df.shape} to {base_df_path}")
base_df.write_csv(base_df_path)

logger.info("End of Tabular Generation")

# check csv
# Construct the full file path
final_file_path = os.path.join(output_dir, output_fname)

# Read the processed CSV file
try:
    final_df = pl.read_csv(final_file_path)
    print("File read successfully.")
    print(final_df.head())
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# 

In [ ]:
# %%
import polars as pl
tab_path = f"./tabular_full/{output_fname}"    
df = pl.read_csv(tab_path)

# Overall row/patient counts
print(f"Rows: {df.shape[0]}, Patients: {df['PatientID'].n_unique()}")

# Prevalence of CKD_stage (row-level and patient-level, since patients can span stages)
print(df["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / df.shape[0] * 100).round(2).alias("pct_rows")
))

# Patient-level prevalence: each patient's max (worst) CKD_stage
patient_stage = df.group_by("PatientID").agg(pl.col("CKD_stage_numeric").max())
print(patient_stage["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / patient_stage.shape[0] * 100).round(2).alias("pct_patients")
))



In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
len(df['PatientID'].unique())